In [25]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator
from langchain_google_genai import ChatGoogleGenerativeAI
import os

In [6]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: str | ForwardRef('os.PathLike[str]') | None = None, stream: IO[str] | None = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: str | None = 'utf-8') -> bool>

In [9]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key=os.getenv("GOOGLE_API_KEY")
)


In [ ]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [12]:
structure_model = model.with_structured_output(EvaluationSchema)

In [29]:
essay = """The Evolution of Artificial Intelligence in Pakistan

Artificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century. It is changing the way people work, learn, communicate, and solve problems. Pakistan, like many developing countries, has gradually recognized the importance of AI and its potential to strengthen the economy, improve public services, create employment opportunities, and address national challenges. The evolution of AI in Pakistan has moved from academic research and small-scale experimentation to a broader national strategy involving government institutions, universities, startups, and the private sector.

In the early stages, AI development in Pakistan was mainly concentrated in universities and research institutions. Computer science departments began introducing areas such as machine learning, robotics, natural language processing, computer vision, and data science. A major development was the establishment of the National Center of Artificial Intelligence (NCAI), which brought together researchers and institutions to work on AI-related research and applications. Over time, Pakistani students and researchers increasingly participated in international AI competitions, research projects, and technology communities. The growth of online learning platforms also made AI education more accessible to Pakistani students.

The development of Pakistan's technology industry further contributed to the growth of AI. Software companies and technology startups began using machine learning and data-driven systems in areas such as finance, healthcare, e-commerce, agriculture, customer service, and cybersecurity. At the same time, Pakistani freelancers and software developers started using modern AI tools to provide services to international clients. The rapid growth of generative AI, particularly after the emergence of tools such as ChatGPT, further increased public awareness of artificial intelligence and encouraged students, developers, entrepreneurs, and businesses to experiment with AI.

The government gradually began treating AI as an important component of Pakistan's digital and economic future. In January 2025, the National AI Taskforce discussed an implementation roadmap focused on AI education, research, innovation, and applications across different sectors. The government identified areas including education, healthcare, agriculture, climate, business, and governance where AI could provide measurable benefits. ([Prime Minister's Office Pakistan][1])

A major milestone in Pakistan's AI journey came in July 2025 with the approval of the **National Artificial Intelligence Policy 2025**, Pakistan's first comprehensive national AI policy. The policy is based on six major pillars: developing an AI innovation ecosystem, increasing awareness and readiness, creating a secure AI ecosystem, transforming important sectors through AI, developing AI infrastructure, and promoting international collaboration. ([Ministry of IT & Telecommunications][2])

One of the most important aspects of the policy is its focus on human capital. Pakistan has a large young population, and developing AI skills among students and professionals could create significant economic opportunities. The policy proposes large-scale training, scholarships, internships, and AI education initiatives. It also proposes Centres of Excellence in AI and a National AI Fund to support research, startups, innovation, and commercialization. ([Ministry of IT & Telecommunications][2])

AI has the potential to transform several important sectors of Pakistan. In **education**, AI can provide personalized learning, intelligent tutoring, automated assessment, and educational content. In **healthcare**, AI can assist doctors in medical diagnosis, medical imaging, disease prediction, and patient management. In **agriculture**, AI can help farmers monitor crops, predict weather conditions, detect diseases, and improve the efficient use of water and fertilizers. In **governance**, AI can improve public services, analyze large amounts of government data, detect fraud, and support better decision-making. The government has specifically identified sectors such as health, education, agriculture, finance, and governance for AI use cases. ([Ministry of IT & Telecommunications][3])

Despite these opportunities, Pakistan faces significant challenges in developing a strong AI ecosystem. One major challenge is the shortage of highly skilled AI professionals. Although many Pakistani students are learning programming, machine learning, and generative AI, advanced research requires strong mathematical knowledge, computing resources, experienced researchers, and access to large datasets. Another challenge is infrastructure. Modern AI systems require powerful computing resources, reliable internet connectivity, large datasets, and substantial investment.

Data privacy and responsible AI are also important concerns. AI systems depend heavily on data, and inappropriate collection or use of personal information can create serious risks. Pakistan therefore needs strong data-protection mechanisms, cybersecurity standards, transparency, and accountability. The National AI Policy recognizes the importance of secure and ethical AI and proposes regulatory sandboxes, cybersecurity measures, and transparency frameworks. ([Ministry of IT & Telecommunications][2])

Another important issue is the possible impact of AI on employment. Pakistan has a large services and outsourcing sector, and automation could affect some routine jobs. However, AI can also create new opportunities in software development, data science, AI engineering, research, cybersecurity, automation, and entrepreneurship. Therefore, instead of viewing AI only as a threat to employment, Pakistan should focus on reskilling its workforce and preparing students for an AI-driven economy.

By 2026, Pakistan's AI journey has begun moving beyond policy formulation toward questions of implementation, governance, and national capability. The **Islamabad AI Declaration**, adopted in February 2026, emphasizes sovereign, responsible, and capability-driven AI. It highlights human accountability, trusted data management, explainable and auditable systems, inclusive innovation, sovereign computing capacity, and private-sector-led development. ([Ministry of IT & Telecommunications][4])

The future of AI in Pakistan will therefore depend not only on adopting foreign AI technologies but also on developing local talent, research capabilities, datasets, infrastructure, and businesses. Pakistan should encourage collaboration among universities, government institutions, technology companies, startups, and international organizations. Investment in AI research and education should be accompanied by policies that protect citizens' rights and encourage innovation.

In conclusion, the evolution of AI in Pakistan represents a transition from limited academic research to a national technological priority. The country has significant challenges, including limited infrastructure, shortage of advanced skills, funding constraints, data governance issues, and the need for effective policy implementation. However, Pakistan also has important advantages, including a young population, a growing technology sector, a large pool of computer science graduates, and an expanding startup and freelance community. The National AI Policy 2025 and subsequent initiatives provide a foundation for future development. If Pakistan successfully invests in people, infrastructure, research, ethical governance, and innovation, AI can become an important tool for economic growth, better public services, and a more competitive position in the global digital economy.
"""

In [14]:
prompt = f'Evaluate the language quality of the following essay and provide a feedbac and assign a score out of 10 \n {essay}'

In [16]:
structure_model.invoke(prompt).feedback

"The essay demonstrates excellent language quality with a clear, formal, and academic tone throughout. The grammar, syntax, and punctuation are accurate, and the vocabulary is well-suited for a policy and technology review essay (e.g., 'sovereign computing capacity', 'regulatory sandboxes', 'transformative'). The progression of ideas is logical, moving seamlessly from historical context and university initiatives to national policies and future challenges. To improve further, the writer could introduce more sentence variety, particularly in the section describing AI's impact on key sectors where the structure 'In [Sector], AI can...' is repeated several times. Overall, it is well-written, articulate, and highly readable."

In [18]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [20]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedbac and assign a score out of 10 \n {state['essay']}'

    output = structure_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [21]:
def evaluate_analysis(state: UPSCState):
    prompt = f'Evaluate the depth of analysis the following essay and provide a feedbac and assign a score out of 10 \n {state['essay']}'
    
    output = structure_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [22]:
def evaluate_thought(state: UPSCState):
    prompt = f'Evaluate the clarity of thought the following essay and provide a feedbac and assign a score out of 10 \n {state['essay']}'
    
    output = structure_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [23]:
def final_evaluation(state: UPSCState):

    #summary feedback

    prompt = f'Based on the following feedbacks create a summarized feedback \n langage feedback - {state['language_feedback']} \n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']}'
    overall_feedback = model.invoke(prompt).content

    #avg feedack
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}



In [27]:
#graph

graph = StateGraph(UPSCState)

#node
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

#edges

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)


workflow = graph.compile()





In [28]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

{'essay': "The Evolution of Artificial Intelligence in Pakistan\n\nArtificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century. It is changing the way people work, learn, communicate, and solve problems. Pakistan, like many developing countries, has gradually recognized the importance of AI and its potential to strengthen the economy, improve public services, create employment opportunities, and address national challenges. The evolution of AI in Pakistan has moved from academic research and small-scale experimentation to a broader national strategy involving government institutions, universities, startups, and the private sector.\n\nIn the early stages, AI development in Pakistan was mainly concentrated in universities and research institutions. Computer science departments began introducing areas such as machine learning, robotics, natural language processing, computer vision, and data science. A major development was the establishment 